In [ ]:
from __future__ import print_function, division
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False


class Conv_block(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(Conv_block, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True))
        self.conv_1x1_output = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1)

    def forward(self, x):
        y = self.conv(x)
        x = self.conv_1x1_output(x)
        return x + y


class Dense_layer(nn.Sequential):
    def __init__(self, in_ch, bottleneck_size, growth_rate):
        super(Dense_layer, self).__init__()
        self.add_module('norm1', nn.BatchNorm2d(in_ch))
        self.add_module('relu1', nn.Hardswish(inplace=True))
        self.add_module('conv1', nn.Conv2d(in_ch, bottleneck_size, kernel_size=1, stride=1, padding=0, bias=True))
        self.add_module('norm2', nn.BatchNorm2d(bottleneck_size))
        self.add_module('relu2', nn.Hardswish(inplace=True))
        self.add_module('conv2', nn.Conv2d(bottleneck_size, growth_rate, kernel_size=3, stride=1, padding=1, bias=True))

    def forward(self, x):
        new_features = super(Dense_layer, self).forward(x)
        return torch.cat([x, new_features], 1)


class Dense_block(nn.Sequential):
    def __init__(self, in_channels, layer_counts, growth_rate):
        super(Dense_block, self).__init__()
        for i in range(layer_counts):
            bottleneck_size = 4 * growth_rate
            layer = Dense_layer(in_channels + i * growth_rate, bottleneck_size, growth_rate)
            self.add_module('denselayer%d' % (i + 1), layer)


class TransitionLayer(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        self.add_module('norm', nn.BatchNorm2d(in_channels))
        self.add_module('relu', nn.Hardswish(inplace=True))
        self.add_module('conv', nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False))


class Conv_pooling(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.ConvPooling = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2, padding=0, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.Hardswish(inplace=True))

    def forward(self, x):
        return self.ConvPooling(x)


class ASPP(nn.Module):
    def __init__(self, in_channel):
        super(ASPP, self).__init__()
        self.atrous_block1 = nn.Conv2d(in_channel, in_channel, kernel_size=1, stride=1)
        self.atrous_block2 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=2, dilation=2)
        self.atrous_block3 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=3, dilation=3)
        self.atrous_block4 = nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=1, padding=4, dilation=4)
        self.conv_1x1_output = nn.Conv2d(in_channel * 4, in_channel, kernel_size=1, stride=1)
        self.ConvPooling = nn.Sequential(
            nn.Conv2d(in_channel, in_channel * 2, kernel_size=2, stride=1, padding=1, dilation=2, bias=True),
            nn.BatchNorm2d(in_channel * 2),
            nn.Hardswish(inplace=True),
            nn.MaxPool2d(2, stride=2),
            nn.BatchNorm2d(in_channel * 2),
            nn.Hardswish(inplace=True))

    def forward(self, x):
        a1 = self.atrous_block1(x)
        a2 = self.atrous_block2(x)
        a3 = self.atrous_block3(x)
        a4 = self.atrous_block4(x)
        out = self.conv_1x1_output(torch.cat([a1, a2, a3, a4], dim=1))
        return self.ConvPooling(out + x)


class CatConv_block(nn.Module):
    def __init__(self, in_ch, growth):
        super(CatConv_block, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch * growth, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True))
        self.Up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear'),
            nn.BatchNorm2d(in_ch * 2),
            nn.Hardswish(inplace=True),
            nn.Conv2d(in_ch * 2, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True))
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_ch * growth, in_ch * growth, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(in_ch * growth),
            nn.Hardswish(inplace=True),
            nn.Conv2d(in_ch * growth, in_ch, kernel_size=1, stride=1),
            nn.BatchNorm2d(in_ch),
            nn.Hardswish(inplace=True))

    def forward(self, x1, x2):
        x2 = self.Up(x2)
        x = torch.cat([x1, x2], dim=1)
        return self.conv2(x) + self.conv1(x)


class Net(nn.Module):
    """
    QHUPnet: Quantum Hologram Undetected Photon Network
    Input  : hologram      [B, 1, 512, 512]
    Output : reconstruction [B, 1, 256, 256]
    """
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.CP    = Conv_pooling(1, 32)
        self.Aspp1 = ASPP(32);   self.Aspp2 = ASPP(64)
        self.Aspp3 = ASPP(128);  self.Aspp4 = ASPP(256); self.Aspp5 = ASPP(512)
        self.conv1x1 = nn.Conv2d(16, 1, kernel_size=1, stride=1)
        self.Conv    = Conv_block(32, 16)
        self.Dense1  = Dense_block(512, 5, 32);  self.Trans1 = TransitionLayer(512+5*32, 512)
        self.Dense2  = Dense_block(256, 5, 16);  self.Trans2 = TransitionLayer(256+5*16, 256)
        self.Dense3  = Dense_block(128, 5, 16);  self.Trans3 = TransitionLayer(128+5*16, 128)
        self.Dense4  = Dense_block(64,  5, 16);  self.Trans4 = TransitionLayer(64 +5*16, 64)
        self.Dense5  = Dense_block(32,  5, 16);  self.Trans5 = TransitionLayer(32 +5*16, 32)
        self.CatConv01 = CatConv_block(32,  2); self.CatConv11 = CatConv_block(64,  2)
        self.CatConv21 = CatConv_block(128, 2); self.CatConv31 = CatConv_block(256, 2)
        self.CatConv41 = CatConv_block(512, 2); self.CatConv02 = CatConv_block(512, 2)
        self.CatConv12 = CatConv_block(256, 2); self.CatConv22 = CatConv_block(128, 2)
        self.CatConv32 = CatConv_block(64,  2); self.CatConv42 = CatConv_block(32,  2)

    def forward(self, x):
        # Encoder
        C00 = self.CP(x)       # (32, 256, 256)
        C10 = self.Aspp1(C00)  # (64, 128, 128)
        C20 = self.Aspp2(C10)  # (128,  64,  64)
        C30 = self.Aspp3(C20)  # (256,  32,  32)
        C40 = self.Aspp4(C30)  # (512,  16,  16)
        C50 = self.Aspp5(C40)  # (1024,  8,   8)
        # Feature fusion — column 1
        C01 = self.CatConv01(C00, C10)
        C11 = self.CatConv11(C10, C20)
        C21 = self.CatConv21(C20, C30)
        C31 = self.CatConv31(C30, C40)
        C41 = self.CatConv41(C40, C50)
        # Decoder with Dense blocks
        C42 = self.Trans1(self.Dense1(self.CatConv02(C41, C50)))
        C32 = self.Trans2(self.Dense2(self.CatConv12(C31, C42)))
        C22 = self.Trans3(self.Dense3(self.CatConv22(C21, C32)))
        C12 = self.Trans4(self.Dense4(self.CatConv32(C11, C22)))
        C02 = self.Trans5(self.Dense5(self.CatConv42(C01, C12)))
        # Output head
        out = self.conv1x1(self.Conv(C02))  # (1, 256, 256)
        return out

print("Model architecture loaded.")
